# Reconnaissance d'entités nommées avec SpaCy

La documentation est accessible ici: https://spacy.io/api

## Imports

In [1]:
import os
from collections import defaultdict
import spacy
from spacy.lang.fr.examples import sentences

In [2]:
nlp = spacy.load('fr_core_news_md')

## Exemple sur un corpus de test fourni par SpaCy

In [3]:
# Imprimer le corpus de Spacy
sentences

['Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars',
 "Les voitures autonomes déplacent la responsabilité de l'assurance vers les constructeurs",
 "San Francisco envisage d'interdire les robots coursiers sur les trottoirs",
 'Londres est une grande ville du Royaume-Uni',
 'L’Italie choisit ArcelorMittal pour reprendre la plus grande aciérie d’Europe',
 "Apple lance HomePod parce qu'il se sent menacé par l'Echo d'Amazon",
 "La France ne devrait pas manquer d'électricité cet été, même en cas de canicule",
 'Nouvelles attaques de Trump contre le maire de Londres',
 'Où es-tu ?',
 'Qui est le président de la France ?',
 'Où est la capitale des États-Unis ?',
 'Quand est né Barack Obama ?']

In [4]:
# Isoler la première phrase
sent = sentences[0]
sent

'Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars'

In [5]:
# Traiter la phrase avec Spacy
doc = nlp(sent)

In [6]:
type(doc)

spacy.tokens.doc.Doc

In [7]:
doc.text

'Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars'

In [8]:
doc.to_json()

{'text': 'Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars',
 'ents': [{'start': 0, 'end': 5, 'label': 'ORG'}],
 'sents': [{'start': 0, 'end': 72}],
 'tokens': [{'id': 0,
   'start': 0,
   'end': 5,
   'tag': 'PROPN',
   'pos': 'PROPN',
   'morph': 'Gender=Masc|Number=Sing',
   'lemma': 'Apple',
   'dep': 'nsubj',
   'head': 1},
  {'id': 1,
   'start': 6,
   'end': 13,
   'tag': 'VERB',
   'pos': 'VERB',
   'morph': 'Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin',
   'lemma': 'cherche',
   'dep': 'ROOT',
   'head': 1},
  {'id': 2,
   'start': 14,
   'end': 15,
   'tag': 'ADP',
   'pos': 'ADP',
   'morph': '',
   'lemma': 'à',
   'dep': 'mark',
   'head': 3},
  {'id': 3,
   'start': 16,
   'end': 23,
   'tag': 'VERB',
   'pos': 'VERB',
   'morph': 'VerbForm=Inf',
   'lemma': 'acheter',
   'dep': 'xcomp',
   'head': 1},
  {'id': 4,
   'start': 24,
   'end': 27,
   'tag': 'DET',
   'pos': 'DET',
   'morph': 'Definite=Ind|Gender=Fem|Number=Sing|PronType=Art'

In [9]:
# Appliquer le test sur toutes les phrases
for sent in sentences:
    doc = nlp(sent)
    entities = []
    for ent in doc.ents:
        entities.append(f"{ent.text} ({ent.label_})")
    if entities:
        print(f"'{doc.text}' contient les entités suivantes : {', '.join(entities)}")
    else:
        print(f"'{doc.text}' ne contient aucune entité")

'Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars' contient les entités suivantes : Apple (ORG)
'Les voitures autonomes déplacent la responsabilité de l'assurance vers les constructeurs' ne contient aucune entité
'San Francisco envisage d'interdire les robots coursiers sur les trottoirs' contient les entités suivantes : San Francisco (LOC)
'Londres est une grande ville du Royaume-Uni' contient les entités suivantes : Londres (LOC), Royaume-Uni (LOC)
'L’Italie choisit ArcelorMittal pour reprendre la plus grande aciérie d’Europe' contient les entités suivantes : L’Italie (LOC), ArcelorMittal (ORG), Europe (LOC)
'Apple lance HomePod parce qu'il se sent menacé par l'Echo d'Amazon' contient les entités suivantes : Apple (ORG), HomePod (MISC), Echo (ORG), Amazon (ORG)
'La France ne devrait pas manquer d'électricité cet été, même en cas de canicule' contient les entités suivantes : La France (LOC)
'Nouvelles attaques de Trump contre le maire de Londres' contient les en

## Appliquer la reconnaissance d'entités nommées sur notre corpus

In [10]:
# Charger les textes de l'année choisie
year = 1960
txt_path = "../../data/txt/"
files = sorted([f for f in os.listdir(txt_path) if f.endswith('.txt') and f"_{year}-" in f])
texts = []
for f in files:
    with open(os.path.join(txt_path, f), 'r', encoding='utf-8') as fh:
        texts.append(fh.read())
len(texts)

100

In [11]:
%%time
# Traiter les textes un par un et compter les personnes, organisations et lieux

people = defaultdict(int)
organisations = defaultdict(int)
places = defaultdict(int)

for doc in nlp.pipe(texts, disable=["morphologizer", "parser", "attribute_ruler", "lemmatizer"]):
    for ent in doc.ents:
        if len(ent.text) > 3:
            if ent.label_ == "PER":
                people[ent.text] += 1
            elif ent.label_ == "ORG":
                organisations[ent.text] += 1
            elif ent.label_ == "LOC":
                places[ent.text] += 1

CPU times: total: 1min 5s
Wall time: 1min 9s


In [12]:
# Entités à exclure : bruit des petites annonces et mots courants mal reconnus
excluded = {
    "rossel", "agence rossel", "agence rossel num", "rossel num", "ecrire ag", "ecag", "s'adr",
    "culs", "curr", "curric", "trav", "atel", "entret", "pr entret", "pr mén", "k dem",
    "exlg", "inst", "jard", "billng", "référ", "brux", "téléph", "- tél", "s.d.b", "i960",
    "bonne", "monsieur", "lundi", "etat", "centre", "firme", "téléphone", "import",
    "promotion", "parc", "employee", "mutuel", "credit", "assurances", "galeries",
    "manœuvres", "intimité", "service du personnel", "s. a.",
}

In [13]:
# Trier et imprimer les 30 entités les plus fréquentes de chaque type

for label, counts in [("Personnes", people), ("Organisations", organisations), ("Lieux", places)]:
    print(f"--- {label} ---")
    sorted_counts = [(e, n) for e, n in sorted(counts.items(), key=lambda kv: kv[1], reverse=True) if e.lower() not in excluded]
    for entity, freq in sorted_counts[:30]:
        print(f"{entity} apparait {freq} fois dans le corpus")

--- Personnes ---
Louise apparait 53 fois dans le corpus
Van Looy apparait 39 fois dans le corpus
Albert apparait 36 fois dans le corpus
Marie apparait 36 fois dans le corpus
Jean apparait 29 fois dans le corpus
O. N. U. apparait 24 fois dans le corpus
Reine apparait 22 fois dans le corpus
Robert apparait 22 fois dans le corpus
général de Gaulle apparait 21 fois dans le corpus
M. Khrouchtchev apparait 21 fois dans le corpus
Roger apparait 20 fois dans le corpus
Janssens apparait 19 fois dans le corpus
A. P. apparait 19 fois dans le corpus
Joseph apparait 17 fois dans le corpus
Molière apparait 17 fois dans le corpus
A.F.P. apparait 17 fois dans le corpus
Brugmann apparait 16 fois dans le corpus
Elisabethville apparait 16 fois dans le corpus
Dumont apparait 15 fois dans le corpus
Daems apparait 15 fois dans le corpus
Louis apparait 15 fois dans le corpus
A. F. P. apparait 15 fois dans le corpus
Mertens apparait 14 fois dans le corpus
Jacques apparait 14 fois dans le corpus
Desmet appara

Exercice: essayez de lister les lieux (LOC) et les organisations (ORG) les plus mentionnées dans le corpus